In [6]:
import numpy as np

def all_close(a_, b_, atol=1e-10, rtol=1e-10):
    # --- compare per level ---
    for k in range(depth + 1):

        x = a_[k]
        y = b_[k]
        if x is None:
            x = np.zeros_like(y)
        delta = x - y
        print(f"level {k} abs: ", abs(delta).max() < atol) if hasattr(delta, "max") else abs(delta) < atol
        rel_dist = (abs(delta) / abs(x)).max() if hasattr(delta, "max") else abs(delta) / abs(x)
        print(f"level {k} rel: ", rel_dist < rtol if rel_dist < rtol else rel_dist)


B, L, d, depth = 1000, 101, 3, 7

np.random.seed(12335)
path_a = np.array(np.random.randn(B, L - 1, d), dtype=np.float64) * (1.0 / (L - 1)) ** 2
path_a = np.concat([np.zeros((B, 1, d)), np.cumsum(path_a, axis=1)], axis=1)

np.random.seed(23466)
path_b = np.array(np.random.randn(B, L - 1, d), dtype=np.float64) * (1.0 / (L - 1)) ** 2
path_b = np.concat([path_a[:, -1:, :], path_b], axis=1)
path_b = np.cumsum(path_b, axis=1)
path_c = np.concatenate([path_a, path_b[:, 1:]], axis=1)

In [7]:
from tensordev import Universal

import numpy as np

NumpyCore = Universal(np)

In [8]:
sig_a = NumpyCore.tensor_path_signature(path_a, trunc=depth)
len(sig_a), [lvl.shape for lvl in sig_a]

(8,
 [(1000, 1),
  (1000, 3),
  (1000, 9),
  (1000, 27),
  (1000, 81),
  (1000, 243),
  (1000, 729),
  (1000, 2187)])

In [9]:
sig_a = NumpyCore.tensor_path_signature(path_a, trunc=depth)
len(sig_a), [lvl.shape for lvl in sig_a]

path_b = np.array(np.random.randn(B, L - 1, d), dtype=np.float64) * (1.0 / (L - 1)) ** 2
path_b = np.concat([path_a[:, -1:, :], path_b], axis=1)
path_b = np.cumsum(path_b, axis=1)
path_c = np.concatenate([path_a, path_b[:, 1:]], axis=1)

sig_b = NumpyCore.tensor_development([path_b], trunc=depth)
sig_c = NumpyCore.tensor_development([path_c], trunc=depth, block_size=100)

sig_c_ = NumpyCore.tensor_product(sig_a, sig_b, trunc=depth)

for i in range(depth):
    assert np.allclose(sig_a[i], sig_c[i][:, 0]) # check if first block gives sig_a
    assert np.allclose(sig_c_[i], sig_c[i][:, 1]) # check if second block gives sig_a \otimes sig_b

In [10]:
from tensordev import Jax
JaxCore = Jax()

In [11]:
sig_a = JaxCore.tensor_path_signature(path_a, trunc=depth)
sig_b = JaxCore.tensor_path_signature(path_b, trunc=depth)
sig_c = JaxCore.tensor_path_signature(path_c, trunc=depth, block_size=100)

In [12]:
sig_c_ = JaxCore.tensor_product(sig_a, sig_b, trunc=depth)

for i in range(depth):
    assert np.allclose(sig_a[i], sig_c[i][:, 0]) # check if first block gives sig_a
    assert np.allclose(sig_c_[i], sig_c[i][:, 1]) # check if second block gives sig_a \otimes sig_b

In [13]:
import signax  
sig_a_sx = signax.signature(path_a, depth=depth)
sig_b_sx = signax.signature(path_b, depth=depth)
sig_c_sx = signax.signature(path_c, depth=depth)

In [14]:
sig_a_sx_ = JaxCore.tensor_from_flat(sig_a_sx, dim=d, insert_zero_level=1.0)
sig_b_sx_ = JaxCore.tensor_from_flat(sig_b_sx, dim=d, insert_zero_level=1.0)
sig_c_sx_ = JaxCore.tensor_from_flat(sig_c_sx, dim=d, insert_zero_level=1.0)

sig_c_sx_prod = JaxCore.tensor_product(sig_a_sx_, sig_b_sx_, trunc=depth)

for i in range(depth):
    assert np.allclose(sig_c_sx_prod[i], sig_c_sx_[i]) # check if second block gives sig_a \otimes sig_b

In [16]:
sig_a = JaxCore.tensor_development([path_a], trunc=depth)
sig_b = JaxCore.tensor_development([path_b], trunc=depth)
sig_c = JaxCore.tensor_development([path_c], trunc=depth, block_size=100)


In [17]:
import time
import statistics as stats
import math

import numpy as np
import jax.numpy as jnp
import jax.random as jr
from jax import config

config.update("jax_enable_x64", True)

import torch
import sigkernel

from tensordev.kernel.free import free_kernel


# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------
dim = 2
batch_x = 32
batch_y = 32
steps = 256
dyadic_order = 1
reps = 10

# integrated OU parameters
horizon = 1.0
theta = 0.5
sigma = 0.15


# ------------------------------------------------------------
# Data: level-1 integrated OU paths, shared across both methods
# ------------------------------------------------------------
def integrated_ou_paths_numpy(seed, *, batch, steps, dim, horizon=1.0, theta=0.5, sigma=0.15):
    rng = np.random.default_rng(seed)
    dt = horizon / steps
    alpha = math.exp(-theta * dt)

    if theta == 0.0:
        beta = sigma * math.sqrt(dt)
    else:
        beta = sigma * math.sqrt((1.0 - math.exp(-2.0 * theta * dt)) / (2.0 * theta))

    noise = rng.standard_normal((steps, batch, dim))
    x = np.zeros((batch, dim), dtype=np.float64)
    u = np.zeros((batch, dim), dtype=np.float64)
    xs = [x.copy()]

    for n in range(steps):
        u_next = alpha * u + beta * noise[n]
        x_next = x + 0.5 * dt * (u + u_next)
        u, x = u_next, x_next
        xs.append(x.copy())

    return np.stack(xs, axis=1)  # (batch, steps+1, dim)


X_np = integrated_ou_paths_numpy(
    1234, batch=batch_x, steps=steps, dim=dim, horizon=horizon, theta=theta, sigma=sigma
)
Y_np = integrated_ou_paths_numpy(
    5678, batch=batch_y, steps=steps, dim=dim, horizon=horizon, theta=theta, sigma=sigma
)

# our solver takes increments
dx = (jnp.asarray(np.diff(X_np, axis=1)),)   # DenseElemFirstOn with one level
dy = (jnp.asarray(np.diff(Y_np, axis=1)),)

# sigkernel takes paths
X_torch = torch.as_tensor(X_np, dtype=torch.float64)
Y_torch = torch.as_tensor(Y_np, dtype=torch.float64)

sigker = sigkernel.SigKernel(sigkernel.LinearKernel(), dyadic_order=dyadic_order)


# ------------------------------------------------------------
# Timed kernels
# ------------------------------------------------------------
def run_ours():
    out = free_kernel(dx, dy, evaluate="terminal", return_fg=False, pairwise=True, backend="scan",
                      dyadic_order=dyadic_order, increment_in=True)
    return np.asarray(out.block_until_ready())


def run_sigkernel():
    out = sigker.compute_Gram(X_torch, Y_torch)
    if out.device.type == "cuda":
        torch.cuda.synchronize()
    return out.detach().cpu().numpy()


# ------------------------------------------------------------
# Warm-up
# ------------------------------------------------------------
_ = run_ours()
_ = run_sigkernel()


# ------------------------------------------------------------
# Benchmark
# ------------------------------------------------------------
def bench(fn, reps=10):
    times = []
    out = None
    for _ in range(reps):
        t0 = time.perf_counter()
        out = fn()
        t1 = time.perf_counter()
        times.append(t1 - t0)
    return {
        "times": times,
        "mean": stats.mean(times),
        "median": stats.median(times),
        "min": min(times),
        "max": max(times),
        "last_output": out,
    }


res_ours = bench(run_ours, reps=reps)
res_sigk = bench(run_sigkernel, reps=reps)

# sanity check
max_abs_err = np.max(np.abs(res_ours["last_output"] - res_sigk["last_output"]))

print(f"shape: {res_ours['last_output'].shape}")
print(f"max |ours - sigkernel| = {max_abs_err:.3e}")
print()

print("ours:")
print(f"  mean   = {res_ours['mean']:.6f} s")
print(f"  median = {res_ours['median']:.6f} s")
print(f"  min    = {res_ours['min']:.6f} s")
print(f"  max    = {res_ours['max']:.6f} s")
print()

print("sigkernel:")
print(f"  mean   = {res_sigk['mean']:.6f} s")
print(f"  median = {res_sigk['median']:.6f} s")
print(f"  min    = {res_sigk['min']:.6f} s")
print(f"  max    = {res_sigk['max']:.6f} s")
print()

print(f"speedup (median, sigkernel / ours) = {res_sigk['median'] / res_ours['median']:.2f}x")

shape: (32, 32)
max |ours - sigkernel| = 1.414e-06

ours:
  mean   = 0.910516 s
  median = 0.924835 s
  min    = 0.787207 s
  max    = 0.996771 s

sigkernel:
  mean   = 5.345659 s
  median = 5.289321 s
  min    = 4.871668 s
  max    = 6.140465 s

speedup (median, sigkernel / ours) = 5.72x


In [8]:
from tensordev import jax as tj

NameError: name 'tensor_product' is not defined

NameError: name 'tensor_product' is not defined